# Notebook 4: Real-Time PL-Accelerated FFT & Spectrum Analyzer Guide

This notebook demonstrates how to capture and visualize frequency spectra computed entirely within the **Programmable Logic (PL)** using the **Xilinx LogiCORE FFT (2048-pt)** and **CORDIC Magnitude Engine** on the PYNQ-Z2.

## 1. System Setup & Permission Check

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
from pynq_oscilloscope.fft_dma import StreamingFFT
import matplotlib.pyplot as plt
import time

# Grant permissions for /dev/bus/usb
check_usb_permissions()

## 2. Load the Hardware Overlay
Instantiate `OscilloscopeOverlay()`. It automatically loads the dual-DMA bitstream containing both Time-Domain and Frequency-Domain hardware engines.

In [ ]:
ol = OscilloscopeOverlay()
print("✅ Oscilloscope & Spectrum Analyzer Overlay loaded successfully!")
print(f"  • Time Packet Size : {ol.packet_size} samples @ 1 MSPS")
print(f"  • PL FFT Bins      : {ol.fft_points // 2} unique frequency bins (0 Hz to 500 kHz)")

## 3. Generate a Test Tone with Analog Discovery 3
Start generating a **25 kHz Sine wave** (1.5V Amplitude, 1.65V DC Offset) on Wavegen Channel 1 (W1 $\rightarrow$ A0).

In [ ]:
# Start 25 kHz Sine wave
ol.wavegen.start(shape="Sine", frequency=25000.0, amplitude=1.5, offset=1.65)
time.sleep(1.0)
print("✅ AD3 active @ 25 kHz (1.5V amplitude, 1.65V offset).")

## 4. Capture PL Hardware FFT Spectrum
Capture the spectrum in **dBV** logarithmic units directly from the FPGA using `ol.capture_fft()`.

In [ ]:
# Capture single-sided spectrum (0 Hz to 500 kHz)
freqs, mags = ol.capture_fft(unit="dBV")

# Detect peak frequency
peak_freq, peak_mag = StreamingFFT.get_peak_frequency(freqs, mags, min_freq_hz=1000.0)
print(f"🎯 Dominant Peak: {peak_freq/1e3:.2f} kHz @ {peak_mag:.1f} dBV")

## 5. Spectrum Visualization with Matplotlib
Plot the frequency spectrum showing the sharp fundamental peak.

In [ ]:
plt.figure(figsize=(10, 4.5), dpi=100)
plt.plot(freqs / 1e3, mags, color="#FF007F", linewidth=1.6, label=f"PL Hardware FFT (Peak: {peak_freq/1e3:.1f} kHz)")
plt.scatter([peak_freq / 1e3], [peak_mag], color="#00FFCC", s=60, zorder=5, label=f"Fundamental: {peak_freq/1e3:.1f} kHz ({peak_mag:.1f} dBV)")

plt.title("Hardware-Accelerated 1 MSPS FFT Spectrum (0 to 100 kHz)", fontsize=12, fontweight="bold")
plt.xlabel("Frequency (kHz)", fontsize=10)
plt.ylabel("Magnitude (dBV)", fontsize=10)
plt.xlim(0, 100)  # Zoom to 0 - 100 kHz
plt.ylim(-100, 0)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 6. Harmonic Distortion Test (Square Wave Fourier Harmonics)
Update AD3 to a **10 kHz Square wave** and observe the fundamental frequency ($f_0$) along with odd harmonics ($3f_0 = 30\,\text{kHz}$, $5f_0 = 50\,\text{kHz}$, $7f_0 = 70\,\text{kHz}$, $9f_0 = 90\,\text{kHz}$).

In [ ]:
# Switch to 10 kHz Square wave
ol.wavegen.update_parameters(shape="Square", frequency=10000.0, amplitude=1.2)
time.sleep(0.5)

freqs_sq, mags_sq = ol.capture_fft(unit="dBV")

plt.figure(figsize=(10, 4.5), dpi=100)
plt.plot(freqs_sq / 1e3, mags_sq, color="#E040FB", linewidth=1.6, label="10 kHz Square Wave Spectrum")

plt.title("Square Wave Odd Harmonics: 10k, 30k, 50k, 70k, 90k Hz", fontsize=12, fontweight="bold")
plt.xlabel("Frequency (kHz)", fontsize=10)
plt.ylabel("Magnitude (dBV)", fontsize=10)
plt.xlim(0, 100)
plt.ylim(-110, 0)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 7. Clean Hardware Shutdown

In [ ]:
ol.close()
print("🔒 Hardware handles closed and CMA memory released.")